In [ ]:
import os
import glob
import sys
from datetime import datetime
import matplotlib.pyplot as plt
import random
import cv2
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Union
from dataclasses import dataclass, asdict
import math

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'Number of available cpu: {os.cpu_count()}')

In [ ]:
!pip install albumentations

# Data Pipeline

In [ ]:
# Standard library imports
import os
import json
import time
import random
import math
from dataclasses import dataclass
from pathlib import Path
from collections import Counter
from typing import Dict, List, Tuple, Optional

# Third-party imports
import numpy as np
import torch
import albumentations as A
import cv2
from PIL import Image, ImageOps

# PyTorch data utilities
from torch.utils.data import Dataset, DataLoader

# Global constants...
TARGET_CATEGORIES = [
    "10_dress",
    "8_skirt",
    "43_ruffle",
    "1_top__t_shirt__sweatshirt",
    "0_shirt__blouse",
    "4_jacket",
    "9_coat",
    "2_sweater",
    "3_cardigan",
    "5_vest",
    "6_pants",
    "7_shorts",
]
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


### 1) Pipeline Configuration
Define tunable configuration for dataset paths, target categories, and sketch sampling ratios.

In [ ]:
# [Removed] PipelineConfig is no longer needed because the dataset is already preprocessed.


### 2) Image Preprocessing Helpers
Prepare utility functions for consistent padding, resizing, and category-wise file indexing.

In [ ]:
# [Removed] list_category_images is no longer needed because the dataset is already preprocessed.


### 3) Build Paired Metadata
Scan categories, match real and sketch files by stem, and assign sketch methods using the configured sampling ratios.

In [ ]:
# [Removed] build_gan_pairs is no longer needed because the dataset is already preprocessed.


### 4) Dataset and DataLoader
Define the PyTorch dataset and a reusable DataLoader builder with safe defaults.

In [ ]:
# Cell 5
import os
from typing import List, Tuple, Optional
import numpy as np
import torch
import albumentations as A
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class SketchToRealGANDataset(Dataset):
    """PyTorch dataset that loads paired sketch-real images."""

    def __init__(
        self,
        rows: List[dict],
        image_size: int = 256,
        apply_augmentation: bool = False,
        flip_prob: float = 0.5,
        crop_scale: Tuple[float, float] = (0.9, 1.0),
        max_translate_ratio: float = 0.08,
        max_rotation_deg: float = 10.0,
        scale_range: Tuple[float, float] = (0.95, 1.05),
        # Thêm các tham số cho việc làm đứt nét Sketch
        sketch_degrade_prob: float = 0.7,
        degrade_holes_range: Tuple[int, int] = (5, 20),
        degrade_size_range: Tuple[int, int] = (10, 40),
    ):
        self.rows = rows
        self.image_size = image_size
        self.apply_augmentation = apply_augmentation
        
        # --- 1. SPATIAL AUGMENTATION (Áp dụng chung cho cả Real và Sketch để giữ align) ---
        max_shift_percent = max_translate_ratio * 100.0
        self._spatial_augment = A.Compose(
            [
                A.HorizontalFlip(p=flip_prob),
            ],
            additional_targets={"real": "image"},
        )

        # --- 2. SKETCH DEGRADATION (Chỉ áp dụng cho Sketch để ép model học nội suy) ---
        self._sketch_only_augment = A.Compose([])

    def _load_img(self, path: str) -> np.ndarray:
        img_bgr = cv2.imread(path)
        if img_bgr is None:
            raise ValueError(f"Could not load image: {path}")
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        if img_rgb.shape[0] != self.image_size or img_rgb.shape[1] != self.image_size:
            img_rgb = cv2.resize(img_rgb, (self.image_size, self.image_size), interpolation=cv2.INTER_LINEAR)
        return img_rgb

    def _apply_spatial_augment_pair(
        self,
        sketch_img: np.ndarray,
        real_img: np.ndarray,
    ) -> Tuple[np.ndarray, np.ndarray]:
        transformed = self._spatial_augment(image=sketch_img, real=real_img)
        return transformed["image"], transformed["real"]

    def _to_tensor(self, img: np.ndarray) -> torch.Tensor:
        arr = img.astype(np.float32) / 127.5 - 1.0
        return torch.from_numpy(arr).permute(2, 0, 1)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        sketch_img = self._load_img(row["sketch_path"])
        real_img = self._load_img(row["real_path"])
        
        if self.apply_augmentation:
            # Bước 1: Augment không gian cho cả 2 ảnh để giữ sự đồng nhất
            sketch_img, real_img = self._apply_spatial_augment_pair(sketch_img, real_img)
            
            # Bước 2: Chỉ làm hỏng (degrade) ảnh Sketch
            sketch_img = self._sketch_only_augment(image=sketch_img)["image"]

        return {
            "sketch": self._to_tensor(sketch_img),
            "real": self._to_tensor(real_img),
            "filename_stem": row["filename_stem"],
        }


def build_gan_dataloader(
    rows: List[dict],
    batch_size: int = 16,
    image_size: int = 256,
    shuffle: bool = True,
    num_workers: Optional[int] = None,
    apply_augmentation: bool = False,
    sketch_degrade_prob: float = 0.7, 
) -> DataLoader:
    """Build a DataLoader from paired metadata rows."""
    if num_workers is None:
        num_workers = min(4, os.cpu_count() if os.cpu_count() is not None else 4)

    dataset = SketchToRealGANDataset(
        rows=rows,
        image_size=image_size,
        apply_augmentation=apply_augmentation,
        sketch_degrade_prob=sketch_degrade_prob,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        persistent_workers=True if num_workers > 0 else False,
        prefetch_factor=2 if num_workers > 0 else None,
    )

### 5) Run Pipeline and Quick Verification
Instantiate config, build pairs, create DataLoader, and run a short dry-run to validate input throughput.

In [ ]:
import json
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm

# =========================================================
# CONFIGURATION FOR PREPROCESSED DATASET
# =========================================================
# Đường dẫn đến thư mục chứa dữ liệu đã được tiền xử lý và đóng gói (từ preprocess_data.ipynb)
# Thư mục này mặc định chứa: train_real/, train_sketch/, và metadata.json
PREPROCESSED_DATA_DIR = "/kaggle/input/datasets/vunhuduc/sketch-preprocessdata"

# =========================================================
# STEP 1: Load preprocessed metadata
# =========================================================
metadata_file = Path(PREPROCESSED_DATA_DIR) / "metadata.json"

print(f"Loading metadata from: {metadata_file}")
with open(metadata_file, "r", encoding="utf-8") as f:
    preprocessed_rows = json.load(f)

# STEP 2: Reconstruct absolute image paths for dataset
gan_rows = []
for row in preprocessed_rows:
    gan_rows.append({
        "category": row["category"],
        "filename_stem": row["filename_stem"],
        "real_path": str(Path(PREPROCESSED_DATA_DIR) / row["real_path"]),
        "sketch_path": str(Path(PREPROCESSED_DATA_DIR) / row["sketch_path"]),
        "sketch_method": row["sketch_method"]
    })

# Print pipeline summary
gan_summary = {
    "num_pairs": len(gan_rows),
    "sketch_method_counts": dict(Counter(r["sketch_method"] for r in gan_rows)),
}

print("\n" + "=" * 30)
print("PIPELINE SUMMARY (LOADED FROM PREPROCESSED DATA):")
print(f"- Total paired samples: {gan_summary['num_pairs']:,}")
for method, count in gan_summary["sketch_method_counts"].items():
    print(f"  + {method}: {count:,} samples")
print("=" * 30)

# =========================================================
# STEP 3: Initialize DataLoader
# =========================================================
print("\nInitializing DataLoader...")
gan_loader = build_gan_dataloader(
    rows=gan_rows,
    batch_size=1,
    image_size=256,
    shuffle=True,
    apply_augmentation=True,
)
print(f"DataLoader ready. Total batches: {len(gan_loader):,}")

# =========================================================
# STEP 4: Dry run verification
# =========================================================
print("\nRunning dry-run for first 10 batches...")
for i, batch in enumerate(
    tqdm(
        gan_loader,
        total=min(10, len(gan_loader)),
        desc="Dry-run",
        unit="batch",
    )
):
    assert batch["sketch"].shape[1:] == (3, 256, 256)
    assert batch["real"].shape[1:] == (3, 256, 256)
    if i >= 9:
        break

print("\nPipeline is ready for training with sketch + z inputs.")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path


def pil_to_uint8(image: Image.Image) -> np.ndarray:
    return np.asarray(image.convert("RGB"), dtype=np.uint8)


def tensor_to_uint8_img(tensor):
    """Convert a normalized CHW tensor in [-1, 1] to a displayable uint8 image."""
    image = tensor.detach().cpu().permute(1, 2, 0).numpy()
    image = ((image + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    return image


def show_before_after_augment(sample_idx: int = 0):
    """Show images before and after augmentation for both sketch and real pairs."""
    if "gan_rows" not in globals() or len(gan_rows) == 0:
        print("gan_rows chưa có dữ liệu. Hãy chạy cell tạo DataLoader trước.")
        return

    if "SketchToRealGANDataset" not in globals():
        print("Dataset class chưa sẵn sàng. Hãy chạy cell Data Pipeline trước.")
        return

    sample = gan_rows[sample_idx]
    before_sketch = pil_to_uint8(Image.open(sample["sketch_path"]).convert("RGB"))
    before_real = pil_to_uint8(Image.open(sample["real_path"]).convert("RGB"))

    preview_dataset = SketchToRealGANDataset(
        rows=[sample],
        image_size=256,
        apply_augmentation=True,
    )
    aug_item = preview_dataset[0]
    after_sketch = tensor_to_uint8_img(aug_item["sketch"])
    after_real = tensor_to_uint8_img(aug_item["real"])

    fig, axes = plt.subplots(2, 2, figsize=(10, 10))

    axes[0, 0].imshow(before_sketch)
    axes[0, 0].set_title("Sketch before augment")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(after_sketch)
    axes[0, 1].set_title("Sketch after augment")
    axes[0, 1].axis("off")

    axes[1, 0].imshow(before_real)
    axes[1, 0].set_title("Real before augment")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(after_real)
    axes[1, 1].set_title("Real after augment")
    axes[1, 1].axis("off")

    plt.suptitle(f"Before vs After Augmentation | sample_idx={sample_idx} | category={sample['category']}")
    plt.tight_layout()
    plt.show()


show_before_after_augment(sample_idx=0)

In [ ]:
# Cell 8
import matplotlib.pyplot as plt
import numpy as np
import random

def visualize_degradation_pipeline(dataset: SketchToRealGANDataset, num_samples: int = 3):
    """
    Trực quan hóa từng bước của quá trình Data Augmentation:
    Original -> Spatial Transform (cả 2) -> Degrade (chỉ Sketch)
    """
    if len(dataset) == 0:
        print("Dataset trống!")
        return

    # Lấy ngẫu nhiên các chỉ số
    indices = random.sample(range(len(dataset)), min(num_samples, len(dataset)))
    
    # Kích thước lưới: num_samples hàng, 5 cột (Real gốc, Sketch gốc, Real Spatial, Sketch Spatial, Sketch Degraded)
    fig, axes = plt.subplots(len(indices), 5, figsize=(18, 3.5 * len(indices)))
    
    # Xử lý trường hợp chỉ có 1 sample (axes sẽ là mảng 1D)
    if len(indices) == 1:
        axes = [axes]

    for i, idx in enumerate(indices):
        row = dataset.rows[idx]
        
        # 1. Load ảnh gốc (đã pad/resize)
        sketch_raw = dataset._load_img(row["sketch_path"])
        real_raw = dataset._load_img(row["real_path"])
        
        # 2. Áp dụng Spatial Augmentation (biến dạng không gian, áp dụng cho cả 2)
        transformed_spatial = dataset._spatial_augment(image=sketch_raw, real=real_raw)
        sketch_spatial = transformed_spatial["image"]
        real_spatial = transformed_spatial["real"]
        
        # 3. Áp dụng Sketch-only Augmentation (chỉ làm đứt nét/méo trên sketch)
        transformed_degrade = dataset._sketch_only_augment(image=sketch_spatial)
        sketch_degraded = transformed_degrade["image"]
        
        # --- Bắt đầu vẽ ---
        panels = [
            (real_raw, f"1. Real (Gốc)\n{row['category']}"),
            (sketch_raw, "2. Sketch (Gốc)"),
            (real_spatial, "3. Real (Sau Xoay/Crop)"),
            (sketch_spatial, "4. Sketch (Sau Xoay/Crop)\nChưa đứt nét"),
            (sketch_degraded, "5. Sketch (Degraded)\nĐã đứt nét & xô lệch"),
        ]
        
        for col, (img, title) in enumerate(panels):
            ax = axes[i][col]
            ax.imshow(img)
            ax.set_title(title, fontsize=11)
            ax.axis("off")
            
            # Thêm viền đỏ cho ảnh Degraded để dễ nhận biết
            if col == 4:
                for spine in ax.spines.values():
                    spine.set_edgecolor('red')
                    spine.set_linewidth(2)
                    spine.set_visible(True)

    plt.suptitle("Step-by-Step Data Augmentation & Sketch Degradation", fontsize=16, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()

# Lấy dataset từ dataloader đã tạo ở Cell 6 (hoặc khởi tạo mới)
# Đảm bảo bạn đã chạy Cell 5 (mới) và Cell 6 trước khi chạy code này
if 'gan_loader' in globals():
    my_dataset = gan_loader.dataset
    visualize_degradation_pipeline(my_dataset, num_samples=3)
else:
    print("Vui lòng khởi tạo 'gan_loader' trước.")

In [ ]:
import matplotlib.pyplot as plt
import random
from pathlib import Path


def preprocess_preview(path: str, target_size: int = 256):
    """Return raw, smart-padded, and resized versions of one image."""
    with Image.open(path) as src_img:
        raw = src_img.convert("RGB")

    w, h = raw.size
    canvas_side = max(w, h, target_size)
    pad_left = (canvas_side - w) // 2
    pad_top = (canvas_side - h) // 2
    pad_right = canvas_side - w - pad_left
    pad_bottom = canvas_side - h - pad_top

    padded = ImageOps.expand(
        raw,
        border=(pad_left, pad_top, pad_right, pad_bottom),
        fill=(255, 255, 255),
    )
    resized = padded.resize((target_size, target_size), Image.BICUBIC)
    return raw, padded, resized


def find_sketch_by_method(row: dict, method: str):
    """Find sketch image path for a specific method using pipeline config roots."""
    if "PIPE_CFG" not in globals():
        return None

    root = PIPE_CFG.sketch_roots.get(method)
    if root is None:
        return None

    category = row["category"]
    stem = row["filename_stem"]
    method_dir = Path(root) / category
    if not method_dir.exists():
        return None

    for ext in IMG_EXTENSIONS:
        candidate = method_dir / f"{stem}{ext}"
        if candidate.exists():
            return str(candidate)
    return None


def tensor_to_uint8_img(t: torch.Tensor) -> np.ndarray:
    """Convert CHW tensor in [-1, 1] to HWC uint8 image."""
    arr = t.detach().cpu().permute(1, 2, 0).numpy()
    arr = ((arr + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
    return arr


def visualize_full_samples(num_samples: int = 3, image_size: int = 256):
    """Visualize full sample information, including all sketch methods and augment result."""
    if "gan_rows" not in globals() or len(gan_rows) == 0:
        print("'gan_rows' chưa có dữ liệu. Hãy chạy Cell 15 trước.")
        return

    if "SketchToRealGANDataset" not in globals():
        print("Dataset class chưa sẵn sàng. Hãy chạy Cell 13 trước.")
        return

    sample_rows = random.sample(gan_rows, k=min(num_samples, len(gan_rows)))

    # Methods come from current config to guarantee complete per-method view.
    method_list = list(PIPE_CFG.sketch_roots.keys()) if "PIPE_CFG" in globals() else []

    # Build one augmented dataset instance to preview synchronized spatial augmentation.
    preview_dataset = SketchToRealGANDataset(
        rows=sample_rows,
        image_size=image_size,
        apply_augmentation=True,
    )

    # Columns: Real(raw/final), Selected Sketch(raw/final), each method final, augmented pair.
    n_cols = 4 + len(method_list) + 2
    fig, axes = plt.subplots(len(sample_rows), n_cols, figsize=(3.2 * n_cols, 3.0 * len(sample_rows)))
    if len(sample_rows) == 1:
        axes = [axes]

    print(f"Hiển thị {len(sample_rows)} mẫu | số cột: {n_cols}")

    for r, row in enumerate(sample_rows):
        # Real + selected sketch processing previews.
        real_raw, _, real_final = preprocess_preview(row["real_path"], target_size=image_size)
        sketch_raw, _, sketch_final = preprocess_preview(row["sketch_path"], target_size=image_size)

        # Augmented aligned pair preview.
        aug_item = preview_dataset[r]
        aug_sketch = tensor_to_uint8_img(aug_item["sketch"])
        aug_real = tensor_to_uint8_img(aug_item["real"])

        panels = [
            (real_raw, f"Real Raw\\n{Path(row['real_path']).name}"),
            (real_final, "Real Final"),
            (sketch_raw, f"Sketch Raw ({row['sketch_method']})"),
            (sketch_final, "Sketch Final (Selected)"),
        ]

        # Add each sketch type (hed/pencil/canny/...) for full comparison.
        for method in method_list:
            m_path = find_sketch_by_method(row, method)
            if m_path is None:
                blank = np.full((image_size, image_size, 3), 255, dtype=np.uint8)
                panels.append((blank, f"{method.upper()} (missing)"))
            else:
                _, _, m_final = preprocess_preview(m_path, target_size=image_size)
                panels.append((m_final, f"{method.upper()}"))

        panels.append((aug_sketch, "Aug Sketch"))
        panels.append((aug_real, "Aug Real"))

        for c, (img, title) in enumerate(panels):
            axes[r][c].imshow(img)
            axes[r][c].set_title(title, fontsize=9)
            axes[r][c].axis("off")

        # Write sample-level text at the first panel for quick traceability.
        axes[r][0].set_ylabel(
            f"Sample {r + 1}\\ncat: {row['category']}",
            rotation=90,
            fontsize=9,
            labelpad=10,
        )

    plt.suptitle(
        "Full Visualization: Real/Selected Sketch/All Sketch Methods/Augmented Pair",
        y=1.02,
        fontsize=14,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()


# Run visualization
visualize_full_samples(num_samples=4, image_size=256)

# UGATIT Architecture Components
This section contains the UGATIT building blocks: UNet Generator, PatchGAN Discriminator, and GANLoss.


In [ ]:
import torch
import torch.nn as nn
from torch.nn.parameter import Parameter
from torch.nn import init
import functools
from torch.optim import lr_scheduler

def init_weights(net, init_type="normal", init_gain=0.02):
    def init_func(m):
        classname = m.__class__.__name__
        if hasattr(m, "weight") and (classname.find("Conv") != -1 or classname.find("Linear") != -1):
            if init_type == "normal":
                init.normal_(m.weight.data, 0.0, init_gain)
            elif init_type == "xavier":
                init.xavier_normal_(m.weight.data, gain=init_gain)
            elif init_type == "kaiming":
                init.kaiming_normal_(m.weight.data, a=0, mode="fan_in")
            elif init_type == "orthogonal":
                init.orthogonal_(m.weight.data, gain=init_gain)
            else:
                raise NotImplementedError("initialization method [%s] is not implemented" % init_type)
            if hasattr(m, "bias") and m.bias is not None:
                init.constant_(m.bias.data, 0.0)
        elif classname.find("BatchNorm2d") != -1:
            init.normal_(m.weight.data, 1.0, init_gain)
            init.constant_(m.bias.data, 0.0)
    print("initialize network with %s" % init_type)
    net.apply(init_func)

def init_net(net, init_type="normal", init_gain=0.02):
    import os
    if torch.cuda.is_available():
        if "LOCAL_RANK" in os.environ:
            local_rank = int(os.environ["LOCAL_RANK"])
            net.to(local_rank)
            print(f"Initialized with device cuda:{local_rank}")
        else:
            net.to(0)
            print("Initialized with device cuda:0")
    init_weights(net, init_type, init_gain=init_gain)
    return net

def define_G(input_nc, output_nc, ngf=64, n_blocks=6, img_size=256, light=True, init_type="normal", init_gain=0.02):
    net = ResnetGenerator(input_nc, output_nc, ngf=ngf, n_blocks=n_blocks, img_size=img_size, light=light)
    return init_net(net, init_type, init_gain)

def define_D(input_nc, ndf=64, n_layers=5, init_type="normal", init_gain=0.02):
    net = Discriminator(input_nc, ndf=ndf, n_layers=n_layers)
    return init_net(net, init_type, init_gain)

class GANLoss(nn.Module):
    def __init__(self, gan_mode, target_real_label=1.0, target_fake_label=0.0):
        super(GANLoss, self).__init__()
        self.register_buffer("real_label", torch.tensor(target_real_label))
        self.register_buffer("fake_label", torch.tensor(target_fake_label))
        self.gan_mode = gan_mode
        if gan_mode == "lsgan":
            self.loss = nn.MSELoss()
        elif gan_mode == "vanilla":
            self.loss = nn.BCEWithLogitsLoss()
        else:
            raise NotImplementedError("gan mode %s not implemented" % gan_mode)

    def get_target_tensor(self, prediction, target_is_real):
        if target_is_real:
            target_tensor = self.real_label
        else:
            target_tensor = self.fake_label
        return target_tensor.expand_as(prediction)

    def __call__(self, prediction, target_is_real):
        target_tensor = self.get_target_tensor(prediction, target_is_real)
        return self.loss(prediction, target_tensor)

# ---------------------------------------------------------
# UGATIT NETWORKS
# ---------------------------------------------------------





class ResnetGenerator(nn.Module):
    def __init__(self, input_nc, output_nc, ngf=64, n_blocks=6, img_size=256, light=False):
        assert(n_blocks >= 0)
        super(ResnetGenerator, self).__init__()
        self.input_nc = input_nc
        self.output_nc = output_nc
        self.ngf = ngf
        self.n_blocks = n_blocks
        self.img_size = img_size
        self.light = light

        DownBlock = []
        DownBlock += [nn.ReflectionPad2d(3),
                      nn.Conv2d(input_nc, ngf, kernel_size=7, stride=1, padding=0, bias=False),
                      nn.InstanceNorm2d(ngf),
                      nn.ReLU(True)]

        # Down-Sampling
        n_downsampling = 2
        for i in range(n_downsampling):
            mult = 2**i
            DownBlock += [nn.ReflectionPad2d(1),
                          nn.Conv2d(ngf * mult, ngf * mult * 2, kernel_size=3, stride=2, padding=0, bias=False),
                          nn.InstanceNorm2d(ngf * mult * 2),
                          nn.ReLU(True)]

        # Down-Sampling Bottleneck
        mult = 2**n_downsampling
        for i in range(n_blocks):
            DownBlock += [ResnetBlock(ngf * mult, use_bias=False)]

        # Class Activation Map
        self.gap_fc = nn.Linear(ngf * mult, 1, bias=False)
        self.gmp_fc = nn.Linear(ngf * mult, 1, bias=False)
        self.conv1x1 = nn.Conv2d(ngf * mult * 2, ngf * mult, kernel_size=1, stride=1, bias=True)
        self.relu = nn.ReLU(True)

        # Gamma, Beta block
        if self.light:
            FC = [nn.Linear(ngf * mult, ngf * mult, bias=False),
                  nn.ReLU(True),
                  nn.Linear(ngf * mult, ngf * mult, bias=False),
                  nn.ReLU(True)]
        else:
            FC = [nn.Linear(img_size // mult * img_size // mult * ngf * mult, ngf * mult, bias=False),
                  nn.ReLU(True),
                  nn.Linear(ngf * mult, ngf * mult, bias=False),
                  nn.ReLU(True)]
        self.gamma = nn.Linear(ngf * mult, ngf * mult, bias=False)
        self.beta = nn.Linear(ngf * mult, ngf * mult, bias=False)

        # Up-Sampling Bottleneck
        for i in range(n_blocks):
            setattr(self, 'UpBlock1_' + str(i+1), ResnetAdaILNBlock(ngf * mult, use_bias=False))

        # Up-Sampling
        UpBlock2 = []
        for i in range(n_downsampling):
            mult = 2**(n_downsampling - i)
            UpBlock2 += [nn.Upsample(scale_factor=2, mode='nearest'),
                         nn.ReflectionPad2d(1),
                         nn.Conv2d(ngf * mult, int(ngf * mult / 2), kernel_size=3, stride=1, padding=0, bias=False),
                         ILN(int(ngf * mult / 2)),
                         nn.ReLU(True)]

        UpBlock2 += [nn.ReflectionPad2d(3),
                     nn.Conv2d(ngf, output_nc, kernel_size=7, stride=1, padding=0, bias=False),
                     nn.Tanh()]

        self.DownBlock = nn.Sequential(*DownBlock)
        self.FC = nn.Sequential(*FC)
        self.UpBlock2 = nn.Sequential(*UpBlock2)

    def forward(self, input):
        x = self.DownBlock(input)

        gap = torch.nn.functional.adaptive_avg_pool2d(x, 1)
        gap_logit = self.gap_fc(gap.view(x.shape[0], -1))
        gap_weight = list(self.gap_fc.parameters())[0]
        gap = x * gap_weight.unsqueeze(2).unsqueeze(3)

        gmp = torch.nn.functional.adaptive_max_pool2d(x, 1)
        gmp_logit = self.gmp_fc(gmp.view(x.shape[0], -1))
        gmp_weight = list(self.gmp_fc.parameters())[0]
        gmp = x * gmp_weight.unsqueeze(2).unsqueeze(3)

        cam_logit = torch.cat([gap_logit, gmp_logit], 1)
        x = torch.cat([gap, gmp], 1)
        x = self.relu(self.conv1x1(x))

        heatmap = torch.sum(x, dim=1, keepdim=True)

        if self.light:
            x_ = torch.nn.functional.adaptive_avg_pool2d(x, 1)
            x_ = self.FC(x_.view(x_.shape[0], -1))
        else:
            x_ = self.FC(x.view(x.shape[0], -1))
        gamma, beta = self.gamma(x_), self.beta(x_)


        for i in range(self.n_blocks):
            x = getattr(self, 'UpBlock1_' + str(i+1))(x, gamma, beta)
        out = self.UpBlock2(x)

        return out, cam_logit, heatmap


class ResnetBlock(nn.Module):
    def __init__(self, dim, use_bias):
        super(ResnetBlock, self).__init__()
        conv_block = []
        conv_block += [nn.ReflectionPad2d(1),
                       nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=0, bias=use_bias),
                       nn.InstanceNorm2d(dim),
                       nn.ReLU(True)]

        conv_block += [nn.ReflectionPad2d(1),
                       nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=0, bias=use_bias),
                       nn.InstanceNorm2d(dim)]

        self.conv_block = nn.Sequential(*conv_block)

    def forward(self, x):
        out = x + self.conv_block(x)
        return out


class ResnetAdaILNBlock(nn.Module):
    def __init__(self, dim, use_bias):
        super(ResnetAdaILNBlock, self).__init__()
        self.pad1 = nn.ReflectionPad2d(1)
        self.conv1 = nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=0, bias=use_bias)
        self.norm1 = adaILN(dim)
        self.relu1 = nn.ReLU(True)

        self.pad2 = nn.ReflectionPad2d(1)
        self.conv2 = nn.Conv2d(dim, dim, kernel_size=3, stride=1, padding=0, bias=use_bias)
        self.norm2 = adaILN(dim)

    def forward(self, x, gamma, beta):
        out = self.pad1(x)
        out = self.conv1(out)
        out = self.norm1(out, gamma, beta)
        out = self.relu1(out)
        out = self.pad2(out)
        out = self.conv2(out)
        out = self.norm2(out, gamma, beta)

        return out + x


class adaILN(nn.Module):
    def __init__(self, num_features, eps=1e-5):
        super(adaILN, self).__init__()
        self.eps = eps
        self.rho = Parameter(torch.Tensor(1, num_features, 1, 1))
        self.rho.data.fill_(0.9)

    def forward(self, input, gamma, beta):
        in_mean, in_var = torch.mean(input, dim=[2, 3], keepdim=True), torch.var(input, dim=[2, 3], keepdim=True)
        out_in = (input - in_mean) / torch.sqrt(in_var + self.eps)
        ln_mean, ln_var = torch.mean(input, dim=[1, 2, 3], keepdim=True), torch.var(input, dim=[1, 2, 3], keepdim=True)
        out_ln = (input - ln_mean) / torch.sqrt(ln_var + self.eps)
        out = self.rho * out_in + (1.0 - self.rho) * out_ln
        out = out * gamma.unsqueeze(2).unsqueeze(3) + beta.unsqueeze(2).unsqueeze(3)

        return out


class ILN(nn.Module):
    def __init__(self, num_features, eps=1e-5):
        super(ILN, self).__init__()
        self.eps = eps
        self.rho = Parameter(torch.Tensor(1, num_features, 1, 1))
        self.gamma = Parameter(torch.Tensor(1, num_features, 1, 1))
        self.beta = Parameter(torch.Tensor(1, num_features, 1, 1))
        self.rho.data.fill_(0.0)
        self.gamma.data.fill_(1.0)
        self.beta.data.fill_(0.0)

    def forward(self, input):
        in_mean, in_var = torch.mean(input, dim=[2, 3], keepdim=True), torch.var(input, dim=[2, 3], keepdim=True)
        out_in = (input - in_mean) / torch.sqrt(in_var + self.eps)
        ln_mean, ln_var = torch.mean(input, dim=[1, 2, 3], keepdim=True), torch.var(input, dim=[1, 2, 3], keepdim=True)
        out_ln = (input - ln_mean) / torch.sqrt(ln_var + self.eps)
        out = self.rho * out_in + (1.0 - self.rho) * out_ln
        out = out * self.gamma + self.beta

        return out


class Discriminator(nn.Module):
    def __init__(self, input_nc, ndf=64, n_layers=5):
        super(Discriminator, self).__init__()
        model = [nn.ReflectionPad2d(1),
                 nn.utils.spectral_norm(
                 nn.Conv2d(input_nc, ndf, kernel_size=4, stride=2, padding=0, bias=True)),
                 nn.LeakyReLU(0.2, True)]

        for i in range(1, n_layers - 2):
            mult = 2 ** (i - 1)
            model += [nn.ReflectionPad2d(1),
                      nn.utils.spectral_norm(
                      nn.Conv2d(ndf * mult, ndf * mult * 2, kernel_size=4, stride=2, padding=0, bias=True)),
                      nn.LeakyReLU(0.2, True)]

        mult = 2 ** (n_layers - 2 - 1)
        model += [nn.ReflectionPad2d(1),
                  nn.utils.spectral_norm(
                  nn.Conv2d(ndf * mult, ndf * mult * 2, kernel_size=4, stride=1, padding=0, bias=True)),
                  nn.LeakyReLU(0.2, True)]

        # Class Activation Map
        mult = 2 ** (n_layers - 2)
        self.gap_fc = nn.utils.spectral_norm(nn.Linear(ndf * mult, 1, bias=False))
        self.gmp_fc = nn.utils.spectral_norm(nn.Linear(ndf * mult, 1, bias=False))
        self.conv1x1 = nn.Conv2d(ndf * mult * 2, ndf * mult, kernel_size=1, stride=1, bias=True)
        self.leaky_relu = nn.LeakyReLU(0.2, True)

        self.pad = nn.ReflectionPad2d(1)
        self.conv = nn.utils.spectral_norm(
            nn.Conv2d(ndf * mult, 1, kernel_size=4, stride=1, padding=0, bias=False))

        self.model = nn.Sequential(*model)

    def forward(self, input):
        x = self.model(input)

        gap = torch.nn.functional.adaptive_avg_pool2d(x, 1)
        gap_logit = self.gap_fc(gap.view(x.shape[0], -1))
        gap_weight = list(self.gap_fc.parameters())[0]
        gap = x * gap_weight.unsqueeze(2).unsqueeze(3)

        gmp = torch.nn.functional.adaptive_max_pool2d(x, 1)
        gmp_logit = self.gmp_fc(gmp.view(x.shape[0], -1))
        gmp_weight = list(self.gmp_fc.parameters())[0]
        gmp = x * gmp_weight.unsqueeze(2).unsqueeze(3)

        cam_logit = torch.cat([gap_logit, gmp_logit], 1)
        x = torch.cat([gap, gmp], 1)
        x = self.leaky_relu(self.conv1x1(x))

        heatmap = torch.sum(x, dim=1, keepdim=True)

        x = self.pad(x)
        out = self.conv(x)

        return out, cam_logit, heatmap


class RhoClipper(object):

    def __init__(self, min, max):
        self.clip_min = min
        self.clip_max = max
        assert min < max

    def __call__(self, module):

        if hasattr(module, 'rho'):
            w = module.rho.data
            w = w.clamp(self.clip_min, self.clip_max)
            module.rho.data = w



## Training Configuration and Utilities

In [ ]:
import csv
import json
import time
import random
from pathlib import Path
from dataclasses import dataclass, asdict
import torch
from torch.optim import Adam
from torch.amp import GradScaler, autocast
import itertools
import matplotlib.pyplot as plt

@dataclass
class TrainConfig:
    # Training
    epochs: int = 100
    batch_size: int = 1
    lr: float = 0.0001
    weight_decay: float = 0.0001
    
    # Model config
    input_nc: int = 3
    output_nc: int = 3
    ngf: int = 64
    ndf: int = 64
    n_blocks: int = 6
    img_size: int = 256
    light: bool = True
    init_type: str = 'normal'
    init_gain: float = 0.02
    
    # Loss weights
    lambda_adv: float = 1.0
    lambda_cycle: float = 10.0
    lambda_identity: float = 10.0
    lambda_cam: float = 1000.0
    gan_mode: str = 'lsgan'
    
    # Logging
    log_every_steps: int = 100
    save_every: int = 10
    sample_every: int = 5
    
    # Mixed precision
    use_amp: bool = True
    
    # Resuming
    resume_checkpoint: str = ""
    resume_strict: bool = True

def get_output_root(run_name: str = None) -> Path:
    base = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./outputs')
    if run_name:
        base = base / run_name
    
    (base / 'checkpoints').mkdir(parents=True, exist_ok=True)
    (base / 'samples').mkdir(parents=True, exist_ok=True)
    return base

def save_sample_grid(real_A, real_B, fake_B, fake_A, rec_A, rec_B, out_path):
    def denorm(t):
        return ((t + 1) * 127.5).clamp(0, 255).byte().cpu().permute(1, 2, 0).numpy()
    
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    
    axes[0, 0].set_title('Real A (Sketch)')
    axes[0, 0].imshow(denorm(real_A[0]))
    axes[0, 0].axis('off')
    
    axes[0, 1].set_title('Fake B (Generated Image)')
    axes[0, 1].imshow(denorm(fake_B[0]))
    axes[0, 1].axis('off')
    
    axes[0, 2].set_title('Rec A (Reconstructed Sketch)')
    axes[0, 2].imshow(denorm(rec_A[0]))
    axes[0, 2].axis('off')

    axes[1, 0].set_title('Real B (Target Image)')
    axes[1, 0].imshow(denorm(real_B[0]))
    axes[1, 0].axis('off')
    
    axes[1, 1].set_title('Fake A (Generated Sketch)')
    axes[1, 1].imshow(denorm(fake_A[0]))
    axes[1, 1].axis('off')
    
    axes[1, 2].set_title('Rec B (Reconstructed Image)')
    axes[1, 2].imshow(denorm(rec_B[0]))
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

def save_checkpoint(out_dir, epoch, netG_A, netG_B, netD_A, netD_B, opt_g, opt_d, scaler, config, filename='last.pt'):
    ckpt = {
        'epoch': epoch,
        'netG_A_state_dict': netG_A.state_dict(),
        'netG_B_state_dict': netG_B.state_dict(),
        'netD_A_state_dict': netD_A.state_dict(),
        'netD_B_state_dict': netD_B.state_dict(),
        'opt_g_state_dict': opt_g.state_dict(),
        'opt_d_state_dict': opt_d.state_dict(),
        'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
        'config': asdict(config),
    }
    torch.save(ckpt, out_dir / 'checkpoints' / filename)



## Training Loop
Custom standalone PyTorch training loop for UGATIT.


In [ ]:
def set_requires_grad(nets, requires_grad=False):
    if not isinstance(nets, list):
        nets = [nets]
    for net in nets:
        if net is not None:
            for param in net.parameters():
                param.requires_grad = requires_grad

def train_ugatit(netG_A, netG_B, netD_A, netD_B, criterionGAN, criterionCycle, criterionIdt, criterionCAM, train_loader, config, out_dir):
    opt_g = Adam(itertools.chain(netG_A.parameters(), netG_B.parameters()), lr=config.lr, betas=(0.5, 0.999), weight_decay=config.weight_decay)
    opt_d = Adam(itertools.chain(netD_A.parameters(), netD_B.parameters()), lr=config.lr, betas=(0.5, 0.999), weight_decay=config.weight_decay)

    scaler = GradScaler('cuda', enabled=(config.use_amp and DEVICE.type == 'cuda'))

    start_epoch = 1
    if config.resume_checkpoint and Path(config.resume_checkpoint).exists():
        print(f"Loading checkpoint from {config.resume_checkpoint}")
        ckpt = torch.load(config.resume_checkpoint, map_location='cpu')
        netG_A.load_state_dict(ckpt['netG_A_state_dict'], strict=config.resume_strict)
        netG_B.load_state_dict(ckpt['netG_B_state_dict'], strict=config.resume_strict)
        netD_A.load_state_dict(ckpt['netD_A_state_dict'], strict=config.resume_strict)
        netD_B.load_state_dict(ckpt['netD_B_state_dict'], strict=config.resume_strict)

        if 'opt_g_state_dict' in ckpt: opt_g.load_state_dict(ckpt['opt_g_state_dict'])
        if 'opt_d_state_dict' in ckpt: opt_d.load_state_dict(ckpt['opt_d_state_dict'])
        if 'scaler_state_dict' in ckpt and ckpt['scaler_state_dict'] is not None:
            scaler.load_state_dict(ckpt['scaler_state_dict'])

        start_epoch = ckpt.get('epoch', 0) + 1
        print(f"Successfully resumed from epoch {start_epoch - 1}")

    global_step = (start_epoch - 1) * len(train_loader)
    
    rho_clipper = RhoClipper(0, 1)

    for epoch in range(start_epoch, config.epochs + 1):
        netG_A.train()
        netG_B.train()
        netD_A.train()
        netD_B.train()

        for batch_idx, batch in enumerate(train_loader):
            real_A = batch['sketch'].to(DEVICE, non_blocking=True)
            real_B = batch['real'].to(DEVICE, non_blocking=True)
            
            # ---------------------------
            # Update Discriminators (D_A and D_B)
            # ---------------------------
            set_requires_grad([netD_A, netD_B], True)
            opt_d.zero_grad(set_to_none=True)
            
            with autocast(device_type='cuda', enabled=scaler.is_enabled()):
                fake_B, _, _ = netG_A(real_A)
                fake_A, _, _ = netG_B(real_B)

                real_B_out, real_B_cam_logit, _ = netD_A(real_B)
                fake_B_out, fake_B_cam_logit, _ = netD_A(fake_B.detach())

                loss_D_A_real = criterionGAN(real_B_out, True) + criterionCAM(real_B_cam_logit, torch.ones_like(real_B_cam_logit))
                loss_D_A_fake = criterionGAN(fake_B_out, False) + criterionCAM(fake_B_cam_logit, torch.zeros_like(fake_B_cam_logit))
                loss_D_A = loss_D_A_real + loss_D_A_fake
                
                real_A_out, real_A_cam_logit, _ = netD_B(real_A)
                fake_A_out, fake_A_cam_logit, _ = netD_B(fake_A.detach())

                loss_D_B_real = criterionGAN(real_A_out, True) + criterionCAM(real_A_cam_logit, torch.ones_like(real_A_cam_logit))
                loss_D_B_fake = criterionGAN(fake_A_out, False) + criterionCAM(fake_A_cam_logit, torch.zeros_like(fake_A_cam_logit))
                loss_D_B = loss_D_B_real + loss_D_B_fake
                
                loss_D = loss_D_A + loss_D_B

            scaler.scale(loss_D).backward()
            scaler.step(opt_d)

            # ---------------------------
            # Update Generators (G_A and G_B)
            # ---------------------------
            set_requires_grad([netD_A, netD_B], False)
            opt_g.zero_grad(set_to_none=True)
            
            with autocast(device_type='cuda', enabled=scaler.is_enabled()):
                fake_B, fake_B_cam_logit, _ = netG_A(real_A)
                rec_A, _, _ = netG_B(fake_B)
                
                fake_A, fake_A_cam_logit, _ = netG_B(real_B)
                rec_B, _, _ = netG_A(fake_A)

                # Adversarial loss
                fake_B_out, fake_B_d_cam_logit, _ = netD_A(fake_B)
                loss_G_A_adv = criterionGAN(fake_B_out, True) + criterionCAM(fake_B_d_cam_logit, torch.ones_like(fake_B_d_cam_logit))
                
                fake_A_out, fake_A_d_cam_logit, _ = netD_B(fake_A)
                loss_G_B_adv = criterionGAN(fake_A_out, True) + criterionCAM(fake_A_d_cam_logit, torch.ones_like(fake_A_d_cam_logit))
                
                # Cycle loss
                loss_cycle_A = criterionCycle(rec_A, real_A)
                loss_cycle_B = criterionCycle(rec_B, real_B)

                # Identity loss
                idt_A, idt_A_cam_logit, _ = netG_A(real_B)
                loss_idt_A = criterionIdt(idt_A, real_B)
                idt_B, idt_B_cam_logit, _ = netG_B(real_A)
                loss_idt_B = criterionIdt(idt_B, real_A)

                # CAM loss for Generators
                loss_cam_G_A = criterionCAM(fake_B_cam_logit, torch.ones_like(fake_B_cam_logit)) + criterionCAM(idt_A_cam_logit, torch.zeros_like(idt_A_cam_logit))
                loss_cam_G_B = criterionCAM(fake_A_cam_logit, torch.ones_like(fake_A_cam_logit)) + criterionCAM(idt_B_cam_logit, torch.zeros_like(idt_B_cam_logit))

                loss_G = (loss_G_A_adv + loss_G_B_adv) * config.lambda_adv +                          (loss_cycle_A + loss_cycle_B) * config.lambda_cycle +                          (loss_idt_A + loss_idt_B) * config.lambda_identity +                          (loss_cam_G_A + loss_cam_G_B) * config.lambda_cam

            scaler.scale(loss_G).backward()
            scaler.step(opt_g)
            
            # Apply Rho clipper
            netG_A.apply(rho_clipper)
            netG_B.apply(rho_clipper)

            scaler.update()
            global_step += 1

            if global_step % config.log_every_steps == 0:
                print(
                    f"Epoch {epoch}/{config.epochs} | "
                    f"Batch {batch_idx + 1}/{len(train_loader)} | "
                    f"Step {global_step} | "
                    f"D_A: {loss_D_A.item():.4f} | "
                    f"D_B: {loss_D_B.item():.4f} | "
                    f"G_A_adv: {loss_G_A_adv.item():.4f} | "
                    f"G_B_adv: {loss_G_B_adv.item():.4f} | "
                    f"Cycle: {(loss_cycle_A + loss_cycle_B).item():.4f} | "
                    f"CAM: {(loss_cam_G_A + loss_cam_G_B).item():.4f}"
                )

        if epoch % config.save_every == 0:
            save_checkpoint(out_dir, epoch, netG_A, netG_B, netD_A, netD_B, opt_g, opt_d, scaler, config, filename='last.pt')
            save_checkpoint(out_dir, epoch, netG_A, netG_B, netD_A, netD_B, opt_g, opt_d, scaler, config, filename=f'epoch_{epoch}.pt')

        if epoch % config.sample_every == 0:
            sample_path = out_dir / 'samples' / f'epoch_{epoch}.png'
            with torch.no_grad():
                netG_A.eval()
                netG_B.eval()
                fake_B_val, _, _ = netG_A(real_A[:1])
                rec_A_val, _, _ = netG_B(fake_B_val)
                fake_A_val, _, _ = netG_B(real_B[:1])
                rec_B_val, _, _ = netG_A(fake_A_val)
                save_sample_grid(real_A[:1], real_B[:1], fake_B_val, fake_A_val, rec_A_val, rec_B_val, sample_path)

        # Clean up CUDA cache after each epoch to prevent fragmentation
        torch.cuda.empty_cache()



## Execution

In [ ]:
if __name__ == '__main__':
    # Initialize Configuration
    config = TrainConfig()

    # Initialize Output Directory
    out_dir = get_output_root(run_name='ugatit_training')
    print(f"Outputs will be saved to: {out_dir}")

    # Build DataLoader
    gan_loader = build_gan_dataloader(
        rows=gan_rows,
        batch_size=config.batch_size,
        image_size=config.img_size,
        shuffle=True,
        apply_augmentation=True,
    )

    # Define networks
    netG_A = define_G(config.input_nc, config.output_nc, config.ngf, config.n_blocks, config.img_size, config.light, config.init_type, config.init_gain)
    netG_B = define_G(config.output_nc, config.input_nc, config.ngf, config.n_blocks, config.img_size, config.light, config.init_type, config.init_gain)
    
    netD_A = define_D(config.output_nc, config.ndf, n_layers=5, init_type=config.init_type, init_gain=config.init_gain)
    netD_B = define_D(config.input_nc, config.ndf, n_layers=5, init_type=config.init_type, init_gain=config.init_gain)

    # Initialize loss functions
    criterionGAN = GANLoss(config.gan_mode).to(DEVICE)
    criterionCycle = torch.nn.L1Loss().to(DEVICE)
    criterionIdt = torch.nn.L1Loss().to(DEVICE)
    criterionCAM = torch.nn.BCEWithLogitsLoss().to(DEVICE)

    # Move networks to device
    netG_A = netG_A.to(DEVICE)
    netG_B = netG_B.to(DEVICE)
    netD_A = netD_A.to(DEVICE)
    netD_B = netD_B.to(DEVICE)

    # Start Training
    print("Starting UGATIT training...")
    train_ugatit(netG_A, netG_B, netD_A, netD_B, criterionGAN, criterionCycle, criterionIdt, criterionCAM, gan_loader, config, out_dir)

